Import

In [1]:
import json
import pandas as pd
import numpy as np

from pathlib import Path
from collections import Counter, defaultdict

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

Config

In [2]:
FILE = "(Copy) FT-data-20260522/Einlagerungstest 080526.json"

# maximum messages to inspect
MAX_MESSAGES = None

Load json

In [3]:
def load_data(file_path):

    try:
        with open(file_path, "r", encoding="utf8") as f:
            data = json.load(f)

        if isinstance(data, list):
            return data

        if isinstance(data, dict):
            return [data]

    except Exception:
        pass

    rows = []

    with open(file_path, "r", encoding="utf8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except:
                pass

    return rows


messages = load_data(FILE)

if MAX_MESSAGES:
    messages = messages[:MAX_MESSAGES]

print("Messages:", len(messages))

Messages: 10070


First look

In [4]:
print(type(messages))

print("\nFirst message:\n")

print(
    json.dumps(
        messages[0],
        indent=2
    )[:5000]
)

<class 'list'>

First message:

{
  "timestamp": "2026-05-08T08:02:58.522435+00:00",
  "topic": "ccu/state/version-mismatch",
  "payload": {
    "timestamp": "2026-05-08T06:36:01.478Z",
    "ccuVersion": "1.3.0",
    "mismatchedModules": []
  }
}


Recursive Field Discovery

In [5]:
field_counter = Counter()

def walk(obj, prefix=""):

    if isinstance(obj, dict):

        for k, v in obj.items():

            name = f"{prefix}.{k}" if prefix else k

            field_counter[name] += 1

            walk(v, name)

    elif isinstance(obj, list):

        for item in obj:

            walk(item, prefix)


for msg in messages:
    walk(msg)

field_df = pd.DataFrame(
    field_counter.items(),
    columns=["field", "count"]
)

field_df = field_df.sort_values(
    "count",
    ascending=False
)

field_df.head(100)

,field,count
0,timestamp,10070
2,payload,10070
1,topic,10070
8,payload.modules.serialNumber,8892
7,payload.modules.type,8892
...,...,...
163,payload.lastNodeId,374
168,payload.batteryState.maxVolt,374
169,payload.driving,374
164,payload.batteryState.minVolt,374


Discover MQTT Topics

In [6]:
possible_topic_fields = [
    "topic",
    "mqtt_topic",
    "metadata.topic"
]

for field in possible_topic_fields:

    if field in field_df.field.values:
        print(field)

topic


Flatten Data

In [7]:
flat = pd.json_normalize(messages)

print(flat.shape)

flat.head()

(10070, 113)


,timestamp,topic,payload.timestamp,payload.ccuVersion,payload.mismatchedModules,payload.modules,payload.intersections,payload.roads,payload.productionDurations.WHITE,payload.productionDurations.BLUE,payload.productionDurations.RED,payload.productionSettings.maxParallelOrders,payload.ftsSettings.chargeThresholdPercent,payload.BLUE.steps,payload.RED.steps,payload.WHITE.steps,payload.ts,payload.stockItems,payload.transports,payload,payload.warehouses,payload.state,payload.br,payload.ldr,payload.hardwareId,payload.hardwareModel,payload.softwareName,payload.softwareVersion,payload.message,payload.t,payload.rt,payload.h,payload.rh,payload.p,payload.iaq,payload.aq,payload.gr,payload.data,payload.connectionState,payload.headerId,payload.version,payload.manufacturer,payload.serialNumber,payload.ip,payload.orderId,payload.actionStates,payload.operatingMode,payload.information,payload.errors,payload.paused,payload.orderUpdateId,payload.actionState.command,payload.actionState.state,payload.actionState.id,payload.actionState.timestamp,payload.protocolFeatures.moduleActions,payload.protocolFeatures.moduleParameters.clearModuleOnPick,payload.typeSpecification.moduleClass,payload.typeSpecification.seriesName,payload.loadSpecification.loadSets,payload.actionState.result,payload.loads,payload.actionState.duration,payload.metadata.opcuaState,payload.actionState.metadata.type,payload.actionState.metadata.workpieceId,payload.loadSpecification.loadPositions,payload.actionState,payload.waitingForLoadHandling,payload.lastNodeSequenceId,payload.nodeStates,payload.lastNodeId,payload.batteryState.minVolt,payload.batteryState.percentage,payload.batteryState.charging,payload.batteryState.currentVoltage,payload.batteryState.maxVolt,payload.driving,payload.lastCode,payload.edgeStates,payload.load,payload.protocolFeatures.agvActions,payload.typeSpecification.navigationTypes,payload.typeSpecification.agvClass,payload.action.id,payload.action.command,payload.action.metadata.type,payload.actions,payload.orderType,payload.type,payload.workpieceId,payload.nodes,payload.edges,payload.productionSteps,payload.receivedAt,payload.startedAt,payload.action.metadata.workpiece.workpieceId,payload.action.metadata.workpiece.type,payload.action.metadata.workpiece.history,payload.action.metadata.workpiece.state,payload.action.metadata.workpieceId,payload.withStorage,payload.calibrating,payload.references,payload.status_references,payload.host,payload.command,payload.position,payload.actionState.metadata.position,payload.actionState.metadata.references,payload.actionState.metadata.green,payload.actionState.metadata.red,payload.actionState.metadata.yellow
0,2026-05-08T08:02:58.522435+00:00,ccu/state/version-mismatch,2026-05-08T06:36:01.478Z,1.3.0,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-05-08T08:02:58.522813+00:00,ccu/state/layout,NaN,NaN,NaN,"[{'type': 'HBW', 'serialNumber': 'SVR4H73901',...","[{'id': '1'}, {'id': '2'}, {'id': '3'}, {'id':...","[{'length': 360, 'from': '1', 'to': '2', 'dire...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-05-08T08:02:58.523407+00:00,ccu/state/config,NaN,NaN,NaN,NaN,NaN,NaN,580.0,560.0,560.0,3.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

Dataset Overview

In [8]:
overview = pd.DataFrame({
    "dtype": flat.dtypes.astype(str),
    "missing": flat.isna().sum(),
    "missing_pct": flat.isna().mean()*100,
    "unique": flat.nunique()
})

overview.sort_values(
    "missing_pct",
    ascending=False
).head(50)

TypeError: unhashable type: 'list'

Identify Numeric Fields

In [9]:
numeric_cols = []

for col in flat.columns:

    try:

        converted = pd.to_numeric(
            flat[col],
            errors="coerce"
        )

        ratio = converted.notna().mean()

        if ratio > 0.8:
            numeric_cols.append(col)

    except:
        pass

print("Numeric columns")

for c in numeric_cols:
    print(c)

Numeric columns


Numeric Statistics

In [10]:
for col in numeric_cols:

    vals = pd.to_numeric(
        flat[col],
        errors="coerce"
    )

    print("\n", "="*80)
    print(col)

    print(vals.describe())

Find Timestamp Fields

In [11]:
timestamp_candidates = []

for col in flat.columns:

    try:

        parsed = pd.to_datetime(
            flat[col],
            errors="coerce"
        )

        success = parsed.notna().mean()

        if success > 0.8:

            timestamp_candidates.append(col)

    except:
        pass

timestamp_candidates

/tmp/ipykernel_20568/492140494.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(
/tmp/ipykernel_20568/492140494.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(
/tmp/ipykernel_20568/492140494.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(
/tmp/ipykernel_20568/492140494.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(
/tmp/ipy

['timestamp']

Categorical Discovery

In [ ]:
for col in flat.columns:

    unique_count = flat[col].nunique(dropna=True)

    if 2 <= unique_count <= 30:

        print("\n")
        print(col)

        print(
            flat[col]
            .value_counts()
            .head(20)
        )

Schema Discovery

In [ ]:
schema_counter = Counter()

for msg in messages:

    keys = sorted(
        list(
            pd.json_normalize(msg).columns
        )
    )

    schema = tuple(keys)

    schema_counter[schema] += 1

print("Unique schemas:", len(schema_counter))

Largest Schemas

In [ ]:
for i, (schema, count) in enumerate(
    schema_counter.most_common(20)
):

    print("\n")
    print("="*80)

    print("Schema", i+1)

    print("Messages:", count)

    for field in schema:
        print(field)

Message Classification

In [ ]:
message_types = defaultdict(int)

for msg in messages:

    cols = set(
        pd.json_normalize(msg)
        .columns
    )

    if "alarm_code" in cols:
        message_types["alarm"] += 1

    elif "temperature" in cols:
        message_types["sensor"] += 1

    elif "order_id" in cols:
        message_types["production"] += 1

    else:
        message_types["other"] += 1

message_types

Topic Analysis

In [ ]:
topic_col = None

for c in flat.columns:

    if "topic" in c.lower():
        topic_col = c
        break

if topic_col:

    (
        flat[topic_col]
        .value_counts()
        .head(50)
        .plot.bar(figsize=(14,6))
    )

    plt.title("MQTT Topics")

    plt.show()

Detect IDs

In [ ]:
for col in flat.columns:

    if "id" in col.lower():

        print("\n")
        print(col)

        print("Unique:", flat[col].nunique())

Detect Constant Fields

In [ ]:
for col in flat.columns:

    if flat[col].nunique(dropna=True) <= 1:

        print(col)

Nested Structures

In [ ]:
nested = []

for msg in messages:

    for k, v in msg.items():

        if isinstance(v, dict):
            nested.append(k)

        if isinstance(v, list):
            nested.append(k)

print(set(nested))

Produce Final Discovery Report

In [ ]:
report = {
    "messages": len(messages),
    "fields": len(flat.columns),
    "schemas": len(schema_counter),
    "numeric_fields": len(numeric_cols),
    "timestamp_fields": len(timestamp_candidates)
}

print(json.dumps(report, indent=2))